In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip -q install "transformers>=4.40,<4.50" "datasets>=2.19" "huggingface_hub>=0.23" scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import random
import shutil
import time
import warnings
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CFG:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    seed = 42
    fp16 = True
    out_dir = "/kaggle/working"
    base_model = "bert-base-uncased"
    temperature = 0.05
    max_seq_len = 32
    eval_max_len = 128
    eval_batch_size = 256
    weight_decay = 0.01
    adam_eps = 1e-8
    warmup_ratio = 0.1
    max_grad_norm = 1.0
    unsup_num_sentences = 1_000_000
    unsup_batch_size = 64
    unsup_lr = 3e-5
    unsup_epochs = 1
    unsup_eval_steps = 250
    sup_batch_size = 256
    sup_lr = 5e-5
    sup_epochs = 3
    sup_eval_steps = 250


set_seed(CFG.seed)
print("Device:", CFG.device, "| torch:", torch.__version__)
assert CFG.device == "cuda", "Enable a GPU accelerator in Kaggle settings, then rerun."

DATA_DIR = "/kaggle/working/simcse_data"
os.makedirs(DATA_DIR, exist_ok=True)


def get_file(repo_id, filename):
    dest = os.path.join(DATA_DIR, filename)
    if not os.path.exists(dest):
        try:
            src = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")
            shutil.copy(src, dest)
        except Exception as err:
            print("hf download failed, using direct url:", err)
            url = f"https://huggingface.co/datasets/{repo_id}/resolve/main/{filename}"
            urllib.request.urlretrieve(url, dest)
    return dest


WIKI_PATH = get_file("princeton-nlp/datasets-for-simcse", "wiki1m_for_simcse.txt")
NLI_PATH = get_file("princeton-nlp/datasets-for-simcse", "nli_for_simcse.csv")

with open(WIKI_PATH, encoding="utf-8") as f:
    WIKI_SENTENCES = [line.strip() for line in f if line.strip()]

nli_df = pd.read_csv(NLI_PATH)
nli_df.columns = [c.strip() for c in nli_df.columns]
nli_df = nli_df.dropna().reset_index(drop=True)
print("wiki sentences:", f"{len(WIKI_SENTENCES):,}", "| nli triplets:", f"{len(nli_df):,}")


def load_sts(repos, splits=("test",)):
    last_error = None
    for repo in repos:
        for split in splits:
            try:
                try:
                    ds = load_dataset(repo, split=split)
                except Exception:
                    ds = load_dataset(repo, split=split, trust_remote_code=True)
                if len(ds) == 0:
                    raise ValueError("empty split")
                return (
                    list(ds["sentence1"]),
                    list(ds["sentence2"]),
                    np.asarray(ds["score"], dtype=np.float64),
                )
            except Exception as err:
                last_error = err
    raise RuntimeError(f"could not load {repos}: {last_error}")


EVAL_TASKS = {
    "STS12": load_sts(["mteb/sts12-sts"]),
    "STS13": load_sts(["mteb/sts13-sts"]),
    "STS14": load_sts(["mteb/sts14-sts"]),
    "STS15": load_sts(["mteb/sts15-sts"]),
    "STS16": load_sts(["mteb/sts16-sts"]),
    "STS-B": load_sts(["mteb/stsbenchmark-sts"]),
    "SICK-R": load_sts(["mteb/sickr-sts"]),
}
STSB_DEV = load_sts(["mteb/stsbenchmark-sts"], splits=("validation", "dev"))
assert len(EVAL_TASKS) == 7
for name, (_, _, sc) in EVAL_TASKS.items():
    print(f"{name:7s}: {len(sc):,} pairs")
print("STS-B dev:", f"{len(STSB_DEV[2]):,} pairs")


class SimCSEModel(nn.Module):
    def __init__(self, base_model=None, encoder=None):
        super().__init__()
        self.encoder = encoder if encoder is not None else AutoModel.from_pretrained(base_model)

    def forward(self, input_ids, attention_mask, use_mlp=True):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if use_mlp and getattr(out, "pooler_output", None) is not None:
            return out.pooler_output
        return out.last_hidden_state[:, 0]


def unsup_contrastive_loss(z1, z2, temperature):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    logits = z1 @ z2.t() / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return F.cross_entropy(logits, labels)


def sup_contrastive_loss(h, h_pos, h_neg, temperature):
    h = F.normalize(h, dim=-1)
    h_pos = F.normalize(h_pos, dim=-1)
    h_neg = F.normalize(h_neg, dim=-1)
    candidates = torch.cat([h_pos, h_neg], dim=0)
    logits = h @ candidates.t() / temperature
    labels = torch.arange(h.size(0), device=h.device)
    return F.cross_entropy(logits, labels)


@torch.no_grad()
def encode_sentences(model, tokenizer, sentences, batch_size=None, max_len=None,
                     use_mlp=True, device=None, fp16=None):
    batch_size = batch_size or CFG.eval_batch_size
    max_len = max_len or CFG.eval_max_len
    device = device or CFG.device
    fp16 = CFG.fp16 if fp16 is None else fp16
    model.eval()
    embs = []
    for i in range(0, len(sentences), batch_size):
        enc = tokenizer(sentences[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.cuda.amp.autocast(enabled=fp16 and device.startswith("cuda")):
            emb = model(enc["input_ids"], enc["attention_mask"], use_mlp=use_mlp)
        embs.append(F.normalize(emb.float(), dim=-1).cpu())
    return torch.cat(embs, dim=0)


def eval_sts_pairs(model, tokenizer, s1, s2, gold_scores, use_mlp):
    e1 = encode_sentences(model, tokenizer, s1, use_mlp=use_mlp)
    e2 = encode_sentences(model, tokenizer, s2, use_mlp=use_mlp)
    sims = (e1 * e2).sum(dim=1).numpy()
    return 100.0 * spearmanr(sims, gold_scores).correlation


def evaluate_stsb_dev(model, tokenizer, use_mlp):
    s1, s2, sc = STSB_DEV
    return eval_sts_pairs(model, tokenizer, s1, s2, sc, use_mlp)


def eval_all_tasks(model, tokenizer, use_mlp, verbose=True):
    results = {}
    for name, (s1, s2, sc) in EVAL_TASKS.items():
        results[name] = eval_sts_pairs(model, tokenizer, s1, s2, sc, use_mlp)
        if verbose:
            print(f"  {name:7s}: {results[name]:.2f}")
    results["Avg."] = float(np.mean([results[k] for k in EVAL_TASKS.keys()]))
    if verbose:
        print(f"  {'Avg.':7s}: {results['Avg.']:.2f}")
    return results


def load_model_from(path_or_name):
    tok = AutoTokenizer.from_pretrained(path_or_name)
    enc = AutoModel.from_pretrained(path_or_name)
    return SimCSEModel(encoder=enc).to(CFG.device), tok


TASK_COLS = ["STS12", "STS13", "STS14", "STS15", "STS16", "STS-B", "SICK-R", "Avg."]
PAPER_UNSUP = [68.40, 82.41, 74.38, 80.91, 78.56, 76.85, 72.23, 76.25]
PAPER_SUP = [75.30, 84.67, 80.19, 85.40, 80.82, 84.25, 80.39, 81.57]


def results_row(res):
    return [round(res[c], 2) for c in TASK_COLS]


def print_comparison(rows):
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", None)
    print(pd.DataFrame(rows, columns=["Model"] + TASK_COLS).to_string(index=False))


def tokenize_in_chunks(tokenizer, texts, max_len, chunk_size=100_000):
    os.environ["TOKENIZERS_PARALLELISM"] = "true"
    id_chunks, mask_chunks = [], []
    try:
        for i in range(0, len(texts), chunk_size):
            enc = tokenizer(texts[i:i + chunk_size], padding="max_length", truncation=True,
                            max_length=max_len, return_tensors="pt")
            id_chunks.append(enc["input_ids"])
            mask_chunks.append(enc["attention_mask"])
    finally:
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
    return torch.cat(id_chunks, 0), torch.cat(mask_chunks, 0)


def average_state_dicts(states):
    avg = {}
    for k, v in states[0].items():
        if v.is_floating_point():
            avg[k] = torch.stack([s[k].float() for s in states], 0).mean(0).to(v.dtype)
        else:
            avg[k] = v.clone()
    return avg


def train_simcse(mode, save_name=None):
    set_seed(CFG.seed)
    save_dir = os.path.join(CFG.out_dir, save_name or f"{mode}-simcse-{CFG.base_model}")
    result_path = os.path.join(save_dir, "dev_result.json")
    if os.path.exists(result_path):
        with open(result_path) as f:
            meta = json.load(f)
        model, tokenizer = load_model_from(save_dir)
        print(f"loaded saved {mode} model, dev = {meta['best_dev_stsb']:.2f}")
        return model, tokenizer, meta["best_dev_stsb"]

    use_fp16 = CFG.fp16 and CFG.device.startswith("cuda")
    tokenizer = AutoTokenizer.from_pretrained(CFG.base_model)
    model = SimCSEModel(base_model=CFG.base_model).to(CFG.device)
    optimizer = scheduler = scaler = best_state = None
    try:
        if mode == "unsup":
            bs, lr, epochs, eval_every = CFG.unsup_batch_size, CFG.unsup_lr, CFG.unsup_epochs, CFG.unsup_eval_steps
            dev_use_mlp = False
            sents = WIKI_SENTENCES
            if len(sents) > CFG.unsup_num_sentences:
                sents = random.Random(CFG.seed).sample(sents, CFG.unsup_num_sentences)
            print(f"tokenizing {len(sents):,} wiki sentences")
            ids, mask = tokenize_in_chunks(tokenizer, sents, CFG.max_seq_len)
            dataset = TensorDataset(ids, mask)
        else:
            bs, lr, epochs, eval_every = CFG.sup_batch_size, CFG.sup_lr, CFG.sup_epochs, CFG.sup_eval_steps
            dev_use_mlp = True
            print(f"tokenizing {len(nli_df):,} nli triplets")
            a0, a1 = tokenize_in_chunks(tokenizer, nli_df["sent0"].astype(str).tolist(), CFG.max_seq_len)
            b0, b1 = tokenize_in_chunks(tokenizer, nli_df["sent1"].astype(str).tolist(), CFG.max_seq_len)
            c0, c1 = tokenize_in_chunks(tokenizer, nli_df["hard_neg"].astype(str).tolist(), CFG.max_seq_len)
            dataset = TensorDataset(a0, a1, b0, b1, c0, c1)

        loader = DataLoader(dataset, batch_size=bs, shuffle=True, drop_last=True,
                            num_workers=2, pin_memory=True)
        total_steps = len(loader) * epochs
        print(f"batch={bs} steps={total_steps:,} lr={lr}")
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                      weight_decay=CFG.weight_decay, eps=CFG.adam_eps)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)

        best_dev, step, t0 = -1.0, 0, time.time()
        for epoch in range(epochs):
            for batch in loader:
                model.train()
                batch = [t.to(CFG.device, non_blocking=True) for t in batch]
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_fp16):
                    if mode == "unsup":
                        input_ids, attn = batch
                        z1 = model(input_ids, attn, use_mlp=True)
                        z2 = model(input_ids, attn, use_mlp=True)
                        loss = unsup_contrastive_loss(z1, z2, CFG.temperature)
                    else:
                        i0, m0, i1, m1, i2, m2 = batch
                        h = model(i0, m0, use_mlp=True)
                        hp = model(i1, m1, use_mlp=True)
                        hn = model(i2, m2, use_mlp=True)
                        loss = sup_contrastive_loss(h, hp, hn, CFG.temperature)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                step += 1
                if step % 200 == 0:
                    print(f"epoch {epoch+1} step {step:,}/{total_steps:,} loss {loss.item():.4f} {time.time()-t0:.0f}s")
                if step % eval_every == 0 or step == total_steps:
                    dev = evaluate_stsb_dev(model, tokenizer, dev_use_mlp)
                    if dev > best_dev:
                        best_dev = dev
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                        print(f"  dev {dev:.2f} at step {step:,} (best)")
                    else:
                        print(f"  dev {dev:.2f} at step {step:,}")

        model.load_state_dict(best_state)
        print(f"done in {(time.time()-t0)/60:.1f} min, best dev = {best_dev:.2f}")
        model.encoder.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        with open(result_path, "w") as f:
            json.dump({"best_dev_stsb": best_dev, "mode": mode, "use_mlp_at_eval": dev_use_mlp}, f)
        torch.cuda.empty_cache()
        return model, tokenizer, best_dev
    except Exception:
        best_state = None
        optimizer = scheduler = scaler = model = None
        gc.collect()
        torch.cuda.empty_cache()
        raise


def train_sup_v2(batch_size=512, lr=5e-5, epochs=3, eval_every=100, soup_k=3,
                 seed=42, gradient_checkpointing=True):
    set_seed(seed)
    use_fp16 = CFG.fp16 and CFG.device.startswith("cuda")
    tokenizer = AutoTokenizer.from_pretrained(CFG.base_model)
    model = SimCSEModel(base_model=CFG.base_model).to(CFG.device)
    if gradient_checkpointing:
        model.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
    optimizer = scheduler = scaler = None
    top = []
    try:
        a0, a1 = tokenize_in_chunks(tokenizer, nli_df["sent0"].astype(str).tolist(), CFG.max_seq_len)
        b0, b1 = tokenize_in_chunks(tokenizer, nli_df["sent1"].astype(str).tolist(), CFG.max_seq_len)
        c0, c1 = tokenize_in_chunks(tokenizer, nli_df["hard_neg"].astype(str).tolist(), CFG.max_seq_len)
        loader = DataLoader(TensorDataset(a0, a1, b0, b1, c0, c1), batch_size=batch_size,
                            shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
        total_steps = len(loader) * epochs
        print(f"batch={batch_size} steps={total_steps:,} lr={lr} checkpointing={gradient_checkpointing}")
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                      weight_decay=CFG.weight_decay, eps=CFG.adam_eps)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)

        step, t0 = 0, time.time()
        for epoch in range(epochs):
            for batch in loader:
                model.train()
                batch = [t.to(CFG.device, non_blocking=True) for t in batch]
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_fp16):
                    i0, m0, i1, m1, i2, m2 = batch
                    h = model(i0, m0, use_mlp=True)
                    hp = model(i1, m1, use_mlp=True)
                    hn = model(i2, m2, use_mlp=True)
                    loss = sup_contrastive_loss(h, hp, hn, CFG.temperature)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                step += 1
                if step % 100 == 0:
                    print(f"epoch {epoch+1} step {step:,}/{total_steps:,} loss {loss.item():.4f} {time.time()-t0:.0f}s")
                if step % eval_every == 0 or step == total_steps:
                    dev = evaluate_stsb_dev(model, tokenizer, True)
                    top.append((dev, step, {k: v.detach().cpu().clone()
                                            for k, v in model.state_dict().items()}))
                    top.sort(key=lambda x: -x[0])
                    del top[soup_k:]
                    print(f"  dev {dev:.2f} at step {step:,} (best {top[0][0]:.2f} @ {top[0][1]})")

        best_single = top[0][0]
        model.load_state_dict(average_state_dicts([s for _, _, s in top]))
        soup_dev = evaluate_stsb_dev(model, tokenizer, True)
        if soup_dev < best_single:
            model.load_state_dict(top[0][2])
            final = best_single
            how = f"single best at step {top[0][1]}"
        else:
            final = soup_dev
            how = f"soup of top {len(top)}"
        print(f"done in {(time.time()-t0)/60:.1f} min, final dev = {final:.2f} ({how})")
        save_dir = os.path.join(CFG.out_dir, "sup-simcse-v2")
        model.encoder.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        with open(os.path.join(save_dir, "result.json"), "w") as f:
            json.dump({"final_dev": final, "how": how, "use_mlp_at_eval": True,
                       "batch_size": batch_size, "lr": lr}, f)
        return model, tokenizer, final
    except Exception:
        top.clear()
        optimizer = scheduler = scaler = model = None
        gc.collect()
        torch.cuda.empty_cache()
        raise

print("setup complete")

Device: cuda | torch: 2.11.0+cu128


wiki1m_for_simcse.txt:   0%|          | 0.00/120M [00:00<?, ?B/s]

nli_for_simcse.csv:   0%|          | 0.00/49.0M [00:00<?, ?B/s]

wiki sentences: 1,000,000 | nli triplets: 275,601


README.md: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

test.jsonl.gz:   0%|          | 0.00/175k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/9927 [00:00<?, ? examples/s]

STS12  : 3,108 pairs
STS13  : 1,500 pairs
STS14  : 3,750 pairs
STS15  : 3,000 pairs
STS16  : 1,186 pairs
STS-B  : 1,379 pairs
SICK-R : 9,927 pairs
STS-B dev: 1,500 pairs
setup complete


In [4]:
unsup_model, unsup_tok, unsup_dev = train_simcse("unsup")
res_unsup = eval_all_tasks(unsup_model, unsup_tok, use_mlp=False)
print_comparison([
    ["paper unsup"] + PAPER_UNSUP,
    ["ours unsup"] + results_row(res_unsup),
])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizing 1,000,000 wiki sentences
batch=64 steps=15,625 lr=3e-05
epoch 1 step 200/15,625 loss 1.5552 41s
  dev 49.45 at step 250 (best)
epoch 1 step 400/15,625 loss 0.1101 83s
  dev 57.50 at step 500 (best)
epoch 1 step 600/15,625 loss 0.0316 126s
  dev 61.79 at step 750 (best)
epoch 1 step 800/15,625 loss 0.0054 169s
epoch 1 step 1,000/15,625 loss 0.0051 210s
  dev 65.46 at step 1,000 (best)
epoch 1 step 1,200/15,625 loss 0.0007 252s
  dev 69.44 at step 1,250 (best)
epoch 1 step 1,400/15,625 loss 0.0010 295s
  dev 73.20 at step 1,500 (best)
epoch 1 step 1,600/15,625 loss 0.0008 337s
  dev 77.16 at step 1,750 (best)
epoch 1 step 1,800/15,625 loss 0.0023 379s
epoch 1 step 2,000/15,625 loss 0.0027 420s
  dev 75.32 at step 2,000
epoch 1 step 2,200/15,625 loss 0.0003 462s
  dev 76.76 at step 2,250
epoch 1 step 2,400/15,625 loss 0.0003 505s
  dev 78.83 at step 2,500 (best)
epoch 1 step 2,600/15,625 loss 0.0003 547s
  dev 79.01 at step 2,750 (best)
epoch 1 step 2,800/15,625 loss 0.0005 589

In [5]:
sup_model, sup_tok, sup_dev = train_simcse("sup")
res_sup = eval_all_tasks(sup_model, sup_tok, use_mlp=True)
print_comparison([
    ["paper sup"] + PAPER_SUP,
    ["ours sup b256"] + results_row(res_sup),
])

tokenizing 275,601 nli triplets
batch=256 steps=3,228 lr=5e-05
epoch 1 step 200/3,228 loss 1.4700 177s
  dev 80.70 at step 250 (best)
epoch 1 step 400/3,228 loss 0.6974 356s
  dev 82.80 at step 500 (best)
epoch 1 step 600/3,228 loss 0.9245 534s
  dev 82.95 at step 750 (best)
epoch 1 step 800/3,228 loss 0.6879 711s
epoch 1 step 1,000/3,228 loss 0.6195 888s
  dev 84.43 at step 1,000 (best)
epoch 2 step 1,200/3,228 loss 0.6128 1066s
  dev 83.88 at step 1,250
epoch 2 step 1,400/3,228 loss 0.6124 1244s
  dev 84.27 at step 1,500
epoch 2 step 1,600/3,228 loss 0.3696 1422s
  dev 84.87 at step 1,750 (best)
epoch 2 step 1,800/3,228 loss 0.4347 1600s
epoch 2 step 2,000/3,228 loss 0.4546 1776s
  dev 84.92 at step 2,000 (best)
epoch 3 step 2,200/3,228 loss 0.4493 1956s
  dev 84.66 at step 2,250
epoch 3 step 2,400/3,228 loss 0.5532 2133s
  dev 84.73 at step 2,500
epoch 3 step 2,600/3,228 loss 0.5211 2311s
  dev 85.00 at step 2,750 (best)
epoch 3 step 2,800/3,228 loss 0.4384 2490s
epoch 3 step 3,000/

In [6]:
for name in ("unsup_model", "unsup_tok", "sup_model", "sup_tok"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

v2_dir = os.path.join(CFG.out_dir, "sup-simcse-v2")
if os.path.exists(os.path.join(v2_dir, "result.json")):
    sup_model2, sup_tok2 = load_model_from(v2_dir)
    with open(os.path.join(v2_dir, "result.json")) as f:
        sup_dev2 = json.load(f)["final_dev"]
    print(f"loaded saved v2 model, dev = {sup_dev2:.2f}")
else:
    sup_model2 = sup_tok2 = sup_dev2 = None
    for bs in (512, 384, 256):
        try:
            print(f"trying batch size {bs}")
            sup_model2, sup_tok2, sup_dev2 = train_sup_v2(batch_size=bs, lr=5e-5, epochs=3, eval_every=100)
            break
        except Exception as err:
            message = str(err)
            recoverable = (isinstance(err, torch.cuda.OutOfMemoryError)
                           or "out of memory" in message.lower()
                           or "CUBLAS_STATUS_ALLOC_FAILED" in message)
            print(f"batch {bs} failed with {type(err).__name__}")
            if not recoverable:
                raise
            err = None
            gc.collect()
            torch.cuda.empty_cache()
    assert sup_model2 is not None, "training failed at every batch size"

res_sup2 = eval_all_tasks(sup_model2, sup_tok2, use_mlp=True)
print(f"v2 dev STS-B: {sup_dev2:.2f}")

rows = [["paper unsup"] + PAPER_UNSUP, ["ours unsup"] + results_row(res_unsup),
        ["paper sup"] + PAPER_SUP, ["ours sup b512+soup"] + results_row(res_sup2)]
if "res_sup" in globals():
    rows.insert(3, ["ours sup b256"] + results_row(res_sup))
print_comparison(rows)

trying batch size 512
batch=512 steps=1,614 lr=5e-05 checkpointing=True
epoch 1 step 100/1,614 loss 2.3332 225s
  dev 76.18 at step 100 (best 76.18 @ 100)
epoch 1 step 200/1,614 loss 1.2048 452s
  dev 81.44 at step 200 (best 81.44 @ 200)
epoch 1 step 300/1,614 loss 1.0705 678s
  dev 82.30 at step 300 (best 82.30 @ 300)
epoch 1 step 400/1,614 loss 0.9150 905s
  dev 83.01 at step 400 (best 83.01 @ 400)
epoch 1 step 500/1,614 loss 0.8854 1131s
  dev 84.13 at step 500 (best 84.13 @ 500)
epoch 2 step 600/1,614 loss 0.7516 1358s
  dev 84.42 at step 600 (best 84.42 @ 600)
epoch 2 step 700/1,614 loss 0.8740 1585s
  dev 84.59 at step 700 (best 84.59 @ 700)
epoch 2 step 800/1,614 loss 0.6380 1812s
  dev 85.07 at step 800 (best 85.07 @ 800)
epoch 2 step 900/1,614 loss 0.6624 2039s
  dev 85.16 at step 900 (best 85.16 @ 900)
epoch 2 step 1,000/1,614 loss 0.6723 2265s
  dev 85.09 at step 1,000 (best 85.16 @ 900)
epoch 3 step 1,100/1,614 loss 0.6483 2492s
  dev 85.03 at step 1,100 (best 85.16 @ 900)


In [7]:
def load_saved(dirname):
    path = os.path.join(CFG.out_dir, dirname)
    meta_path = None
    for name in ("dev_result.json", "result.json"):
        candidate = os.path.join(path, name)
        if os.path.exists(candidate):
            meta_path = candidate
            break
    if meta_path is None:
        print(f"not found on disk, skipping: {dirname}")
        return None
    with open(meta_path) as f:
        meta = json.load(f)
    model, tokenizer = load_model_from(path)
    dev = meta.get("final_dev", meta.get("best_dev_stsb", -1.0))
    return {"model": model, "tokenizer": tokenizer, "dev": dev,
            "use_mlp": meta.get("use_mlp_at_eval", True)}


models = {}
for key, dirname in [("unsup", "unsup-simcse-bert-base-uncased"),
                     ("unsup_seed123", "unsup-simcse-seed123"),
                     ("sup_v1", "sup-simcse-bert-base-uncased"),
                     ("sup_v2", "sup-simcse-v2")]:
    entry = load_saved(dirname)
    if entry is not None:
        models[key] = entry
        print(f"loaded {key}: dev {entry['dev']:.2f}")

results = {}
for key, entry in models.items():
    print(f"evaluating {key} on all 7 tasks")
    results[key] = eval_all_tasks(entry["model"], entry["tokenizer"],
                                  entry["use_mlp"], verbose=False)

rows = [["paper unsup"] + PAPER_UNSUP]
if "unsup" in results:
    rows.append(["ours unsup seed 42"] + results_row(results["unsup"]))
if "unsup_seed123" in results:
    rows.append(["ours unsup seed 123"] + results_row(results["unsup_seed123"]))
    if "unsup" in results:
        rows.append(["ours unsup mean of 2 seeds"]
                    + [round((results["unsup"][c] + results["unsup_seed123"][c]) / 2, 2)
                       for c in TASK_COLS])
paper_sup_added = False
if "sup_v1" in results:
    rows.append(["paper sup"] + PAPER_SUP)
    rows.append(["ours sup b256"] + results_row(results["sup_v1"]))
    paper_sup_added = True
if "sup_v2" in results:
    if not paper_sup_added:
        rows.append(["paper sup"] + PAPER_SUP)
    rows.append(["ours sup b512+soup"] + results_row(results["sup_v2"]))
print_comparison(rows)


@torch.no_grad()
def align_uniform(model, tokenizer, use_mlp, pos_thresh=0.8, n_unif=5000):
    s1, s2, sc = EVAL_TASKS["STS-B"]
    e1 = encode_sentences(model, tokenizer, s1, use_mlp=use_mlp).to(CFG.device)
    e2 = encode_sentences(model, tokenizer, s2, use_mlp=use_mlp).to(CFG.device)
    pos = torch.from_numpy(sc).to(CFG.device) >= pos_thresh
    align = (e1[pos] - e2[pos]).norm(dim=1).pow(2).mean().item()
    uniq = list(dict.fromkeys(list(s1) + list(s2)))[:n_unif]
    e = encode_sentences(model, tokenizer, uniq, use_mlp=use_mlp).to(CFG.device)
    d2 = 2.0 - 2.0 * (e @ e.t())
    off = d2[~torch.eye(d2.size(0), dtype=torch.bool, device=CFG.device)]
    unif = torch.log(torch.exp(-2.0 * off).mean()).item()
    del e1, e2, e, d2, off
    torch.cuda.empty_cache()
    return align, unif


@torch.no_grad()
def retrieve(model, tokenizer, query, corpus, use_mlp, topk=3):
    q = encode_sentences(model, tokenizer, [query], use_mlp=use_mlp).to(CFG.device)
    c = encode_sentences(model, tokenizer, corpus, use_mlp=use_mlp).to(CFG.device)
    picked = (c @ q.t()).squeeze(1).topk(topk).indices.tolist()
    del q, c
    torch.cuda.empty_cache()
    return [corpus[i] for i in picked]


print("alignment / uniformity, lower is better:")
vanilla = SimCSEModel(base_model=CFG.base_model).to(CFG.device)
vanilla_tok = AutoTokenizer.from_pretrained(CFG.base_model)
analysis = [("vanilla bert raw cls", vanilla, vanilla_tok, False)]
if "unsup" in models:
    analysis.append(("unsup simcse", models["unsup"]["model"],
                     models["unsup"]["tokenizer"], False))
if "unsup_seed123" in models:
    analysis.append(("unsup simcse seed123", models["unsup_seed123"]["model"],
                     models["unsup_seed123"]["tokenizer"], False))
if "sup_v2" in models:
    analysis.append(("sup simcse v2", models["sup_v2"]["model"],
                     models["sup_v2"]["tokenizer"], True))
elif "sup_v1" in models:
    analysis.append(("sup simcse b256", models["sup_v1"]["model"],
                     models["sup_v1"]["tokenizer"], True))
for tag, mdl, tk, mlp in analysis:
    a, u = align_uniform(mdl, tk, mlp)
    print(f"  {tag:24s} align={a:.3f} uniform={u:.3f}")
del vanilla
gc.collect()
torch.cuda.empty_cache()

demo = models.get("sup_v2") or models.get("sup_v1")
if demo is not None:
    short_corpus = [s for s in random.Random(0).sample(WIKI_SENTENCES, 200_000)
                    if len(s.split()) <= 10][:20_000]
    for q in ["A man is playing a guitar on stage.", "Two dogs are running in the park."]:
        print("query:", q)
        print(" ", retrieve(demo["model"], demo["tokenizer"], q, short_corpus, demo["use_mlp"]))
else:
    print("no supervised checkpoint on disk, skipping retrieval demo")

models.clear()
gc.collect()
torch.cuda.empty_cache()

loaded unsup: dev 83.28
not found on disk, skipping: unsup-simcse-seed123
loaded sup_v1: dev 85.09
loaded sup_v2: dev 85.43
evaluating unsup on all 7 tasks
evaluating sup_v1 on all 7 tasks
evaluating sup_v2 on all 7 tasks
             Model  STS12  STS13  STS14  STS15  STS16  STS-B  SICK-R  Avg.
       paper unsup  68.40  82.41  74.38  80.91  78.56  76.85   72.23 76.25
ours unsup seed 42  70.55  82.97  74.46  82.05  78.59  78.40   72.21 77.04
         paper sup  75.30  84.67  80.19  85.40  80.82  84.25   80.39 81.57
     ours sup b256  75.05  84.16  79.55  84.67  79.62  83.14   80.20 80.92
ours sup b512+soup  74.80  84.38  79.27  84.61  79.39  83.25   80.40 80.87
alignment / uniformity, lower is better:
  vanilla bert raw cls     align=0.190 uniform=-1.005
  unsup simcse             align=0.355 uniform=-2.320
  sup simcse v2            align=0.528 uniform=-3.716
query: A man is playing a guitar on stage.
  ['He currently can play guitar, mandolin, đàn nhị, đàn nguyệt.', 'Slack-key guit

In [8]:
def train_unsup_v2(batch_size=64, lr=3e-5, epochs=1, eval_every=250, soup_k=3, seed=42):
    set_seed(seed)
    use_fp16 = CFG.fp16 and CFG.device.startswith("cuda")
    tokenizer = AutoTokenizer.from_pretrained(CFG.base_model)
    model = SimCSEModel(base_model=CFG.base_model).to(CFG.device)
    optimizer = scheduler = scaler = None
    top = []
    try:
        sents = WIKI_SENTENCES
        if len(sents) > CFG.unsup_num_sentences:
            sents = random.Random(seed).sample(sents, CFG.unsup_num_sentences)
        print(f"tokenizing {len(sents):,} wiki sentences")
        ids, mask = tokenize_in_chunks(tokenizer, sents, CFG.max_seq_len)
        loader = DataLoader(TensorDataset(ids, mask), batch_size=batch_size, shuffle=True,
                            drop_last=True, num_workers=2, pin_memory=True)
        total_steps = len(loader) * epochs
        print(f"batch={batch_size} steps={total_steps:,} lr={lr}")
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                      weight_decay=CFG.weight_decay, eps=CFG.adam_eps)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)
        step, t0 = 0, time.time()
        for epoch in range(epochs):
            for batch in loader:
                model.train()
                input_ids, attn = [t.to(CFG.device, non_blocking=True) for t in batch]
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_fp16):
                    z1 = model(input_ids, attn, use_mlp=True)
                    z2 = model(input_ids, attn, use_mlp=True)
                    loss = unsup_contrastive_loss(z1, z2, CFG.temperature)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                step += 1
                if step % 200 == 0:
                    print(f"epoch {epoch+1} step {step:,}/{total_steps:,} loss {loss.item():.4f} {time.time()-t0:.0f}s")
                if step % eval_every == 0 or step == total_steps:
                    dev = evaluate_stsb_dev(model, tokenizer, False)
                    top.append((dev, step, {k: v.detach().cpu().clone()
                                            for k, v in model.state_dict().items()}))
                    top.sort(key=lambda x: -x[0])
                    del top[soup_k:]
                    print(f"  dev {dev:.2f} at step {step:,} (best {top[0][0]:.2f} @ {top[0][1]})")
        best_single = top[0][0]
        model.load_state_dict(average_state_dicts([s for _, _, s in top]))
        soup_dev = evaluate_stsb_dev(model, tokenizer, False)
        if soup_dev < best_single:
            model.load_state_dict(top[0][2])
            final = best_single
            how = f"single best at step {top[0][1]}"
        else:
            final = soup_dev
            how = f"soup of top {len(top)}"
        print(f"done in {(time.time()-t0)/60:.1f} min, final dev = {final:.2f} ({how})")
        save_dir = os.path.join(CFG.out_dir, "unsup-simcse-v2")
        model.encoder.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        with open(os.path.join(save_dir, "result.json"), "w") as f:
            json.dump({"final_dev": final, "how": how, "use_mlp_at_eval": False}, f)
        return model, tokenizer, final
    except Exception:
        top.clear()
        optimizer = scheduler = scaler = model = None
        gc.collect()
        torch.cuda.empty_cache()
        raise


u2_model, u2_tok, u2_dev = train_unsup_v2(seed=42)
res_u2 = eval_all_tasks(u2_model, u2_tok, use_mlp=False)
rows = [["paper unsup"] + PAPER_UNSUP]
if "res_unsup" in globals():
    rows.append(["ours unsup seed 42 plain"] + results_row(res_unsup))
rows.append(["ours unsup seed 42 soup"] + results_row(res_u2))
print_comparison(rows)
del u2_model, u2_tok
gc.collect()
torch.cuda.empty_cache()

tokenizing 1,000,000 wiki sentences
batch=64 steps=15,625 lr=3e-05
epoch 1 step 200/15,625 loss 1.5552 41s
  dev 49.45 at step 250 (best 49.45 @ 250)
epoch 1 step 400/15,625 loss 0.1101 84s
  dev 57.50 at step 500 (best 57.50 @ 500)
epoch 1 step 600/15,625 loss 0.0316 127s
  dev 61.79 at step 750 (best 61.79 @ 750)
epoch 1 step 800/15,625 loss 0.0054 169s
epoch 1 step 1,000/15,625 loss 0.0051 210s
  dev 65.46 at step 1,000 (best 65.46 @ 1000)
epoch 1 step 1,200/15,625 loss 0.0007 253s
  dev 69.44 at step 1,250 (best 69.44 @ 1250)
epoch 1 step 1,400/15,625 loss 0.0010 295s
  dev 73.20 at step 1,500 (best 73.20 @ 1500)
epoch 1 step 1,600/15,625 loss 0.0008 337s
  dev 77.16 at step 1,750 (best 77.16 @ 1750)
epoch 1 step 1,800/15,625 loss 0.0023 380s
epoch 1 step 2,000/15,625 loss 0.0027 420s
  dev 75.32 at step 2,000 (best 77.16 @ 1750)
epoch 1 step 2,200/15,625 loss 0.0003 463s
  dev 76.76 at step 2,250 (best 77.16 @ 1750)
epoch 1 step 2,400/15,625 loss 0.0003 505s
  dev 78.83 at step 2,

In [9]:
PAPER_UNSUP_RB = [70.16, 81.77, 73.24, 81.36, 80.65, 80.22, 68.56, 76.57]
PAPER_SUP_RB = [76.53, 85.21, 80.95, 86.03, 82.57, 85.83, 80.50, 82.52]

CFG.base_model = "roberta-base"
CFG.unsup_batch_size = 256
CFG.unsup_lr = 1e-5
CFG.sup_batch_size = 512
CFG.sup_lr = 5e-5

rb_unsup_model, rb_unsup_tok, rb_unsup_dev = train_simcse("unsup", save_name="rb-unsup-simcse")
res_rb_unsup = eval_all_tasks(rb_unsup_model, rb_unsup_tok, use_mlp=False)

rb_sup_model, rb_sup_tok, rb_sup_dev = train_sup_v2(batch_size=512, lr=5e-5, epochs=3, eval_every=100)
res_rb_sup = eval_all_tasks(rb_sup_model, rb_sup_tok, use_mlp=True)
shutil.move("/kaggle/working/sup-simcse-v2", "/kaggle/working/rb-sup-simcse-v2")

print_comparison([
    ["paper unsup roberta"] + PAPER_UNSUP_RB,
    ["ours unsup roberta"] + results_row(res_rb_unsup),
    ["paper sup roberta"] + PAPER_SUP_RB,
    ["ours sup roberta"] + results_row(res_rb_sup),
])
CFG.base_model = "bert-base-uncased"

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizing 1,000,000 wiki sentences
batch=256 steps=3,906 lr=1e-05
epoch 1 step 200/3,906 loss 0.0040 126s
  dev 82.18 at step 250 (best)
epoch 1 step 400/3,906 loss 0.0050 252s
  dev 83.24 at step 500 (best)
epoch 1 step 600/3,906 loss 0.0006 377s
  dev 83.46 at step 750 (best)
epoch 1 step 800/3,906 loss 0.0004 503s
epoch 1 step 1,000/3,906 loss 0.0003 626s
  dev 83.84 at step 1,000 (best)
epoch 1 step 1,200/3,906 loss 0.0003 752s
  dev 83.75 at step 1,250
epoch 1 step 1,400/3,906 loss 0.0004 877s
  dev 83.73 at step 1,500
epoch 1 step 1,600/3,906 loss 0.0003 1001s
  dev 83.45 at step 1,750
epoch 1 step 1,800/3,906 loss 0.0002 1126s
epoch 1 step 2,000/3,906 loss 0.0002 1249s
  dev 83.20 at step 2,000
epoch 1 step 2,200/3,906 loss 0.0002 1374s
  dev 82.96 at step 2,250
epoch 1 step 2,400/3,906 loss 0.0002 1499s
  dev 83.12 at step 2,500
epoch 1 step 2,600/3,906 loss 0.0002 1624s
  dev 83.24 at step 2,750
epoch 1 step 2,800/3,906 loss 0.0002 1749s
epoch 1 step 3,000/3,906 loss 0.0002 1

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


batch=512 steps=1,614 lr=5e-05 checkpointing=True
epoch 1 step 100/1,614 loss 1.1236 227s
  dev 84.86 at step 100 (best 84.86 @ 100)
epoch 1 step 200/1,614 loss 0.9757 454s
  dev 86.03 at step 200 (best 86.03 @ 200)
epoch 1 step 300/1,614 loss 0.8945 681s
  dev 86.74 at step 300 (best 86.74 @ 300)
epoch 1 step 400/1,614 loss 0.7828 909s
  dev 86.86 at step 400 (best 86.86 @ 400)
epoch 1 step 500/1,614 loss 0.7353 1136s
  dev 87.40 at step 500 (best 87.40 @ 500)
epoch 2 step 600/1,614 loss 0.7067 1363s
  dev 86.93 at step 600 (best 87.40 @ 500)
epoch 2 step 700/1,614 loss 0.6905 1592s
  dev 87.30 at step 700 (best 87.40 @ 500)
epoch 2 step 800/1,614 loss 0.6292 1819s
  dev 86.96 at step 800 (best 87.40 @ 500)
epoch 2 step 900/1,614 loss 0.7488 2047s
  dev 87.09 at step 900 (best 87.40 @ 500)
epoch 2 step 1,000/1,614 loss 0.5381 2275s
  dev 87.19 at step 1,000 (best 87.40 @ 500)
epoch 3 step 1,100/1,614 loss 0.5680 2503s
  dev 87.36 at step 1,100 (best 87.40 @ 500)
epoch 3 step 1,200/1,6